conexión con sentinel-2 y datos


In [10]:
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
%pip install sentinelhub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
%pip install rasterio

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from datetime import datetime

from sentinelhub import (
    SHConfig,
    CRS,
    BBox,
    DataCollection,
    SentinelHubRequest,
    MimeType,
    bbox_to_dimensions
)

# para datos espaciales
import rasterio
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt


In [14]:
import os
from dotenv import load_dotenv
from sentinelhub import SHConfig

load_dotenv()

config = SHConfig()

config.sh_client_id = os.getenv("SH_CLIENT_ID")
config.sh_client_secret = os.getenv("SH_CLIENT_SECRET")

print("Client ID configurado:", bool(config.sh_client_id))
print("Client Secret configurado:", bool(config.sh_client_secret))

Client ID configurado: True
Client Secret configurado: True


#### coordenadas de los lagos

In [15]:
# coordenadas de los lagos que están en el lab 
lagos = {
    'Atitlán': {
        'west': -91.326256 ,
        'east': -91.07151 ,
        'south': 14.5948 ,
        'north': 14.750979
    },

    'Amatitlán': {
        'west': -90.638065 ,
        'east': -90.512924 ,
        'south': 14.412347 ,
        'north': 14.493799
    }
}

# fechas dadas en el lab
fechas_atitlan = [ '2025-01-18', '2025-04-13', '2025-05-13', '2025-07-17', '2025-11-21', '2025-12-29', '2026-02-12', '2026-03-24', '2026-04-13', '2026-04-28', '2026-07-22' ]

fechas_amatitlan = [ '2025-01-28', '2025-04-15', '2025-04-28', '2025-11-24', '2026-01-08', '2026-02-02', '2026-02-07', '2026-03-29', '2026-04-13', '2026-04-28', '2026-06-19' ]

fechas = { 'Atitlán': fechas_atitlan, 'Amatitlán': fechas_amatitlan }

print("Lagos definidos:")
for lago, coords in lagos.items(): print(f"  {lago}: {coords}")
print(f"\nFechas por lago:")
for lago, fecha_list in fechas.items(): print(f"  {lago}: {len(fecha_list)} fechas")

Lagos definidos:
  Atitlán: {'west': -91.326256, 'east': -91.07151, 'south': 14.5948, 'north': 14.750979}
  Amatitlán: {'west': -90.638065, 'east': -90.512924, 'south': 14.412347, 'north': 14.493799}

Fechas por lago:
  Atitlán: 11 fechas
  Amatitlán: 11 fechas


directorio para guardar datos

In [16]:
# Crear carpeta para guardar datos descargados
data_dir = Path('datos_sentinel')
data_dir.mkdir(exist_ok=True)

# Subcarpetas para cada lago
for lago in lagos.keys():
    lago_dir = data_dir / lago
    lago_dir.mkdir(exist_ok=True)

print(f"Directorio de datos: {data_dir.absolute()}")
print(f"Subdirectorios creados para cada lago")

Directorio de datos: c:\Users\belen\Documents\Data Science\DataS_Lab4_GeoEspaciales\datos_sentinel
Subdirectorios creados para cada lago


#### ddescargar datos de Sentinel-2

aquí bandas necesarias para calcular 
NDVI: B04 (Rojo) y B08 (NIR - Near Infrared)
NDWI: B03 (Verde) y B08 (NIR)
Cianobacteria



Descarga bandas específicas de Sentinel-2 para un lago en una fecha determinada.
    
    Args:
        bbox (BBox): Caja delimitadora con coordenadas del lago
        fecha (str): Fecha en formato 'YYYY-MM-DD'
        lago_nombre (str): Nombre del lago
        config: Configuración de Sentinel Hub
    
    Returns:
        dict: Diccionario con arrays NumPy de cada banda (B03, B04, B08)

In [20]:
def descargar_bandas_sentinelhub(bbox, fecha, lago_nombre, config=None):

    # las bandas B03 (verde), B04 (rojo), B08 (NIR)
    # SCL para máscara de nubes
    request = SentinelHubRequest(
        evalscript="""
            //VERSION=3
            function setup() {
                return {
                    input: [{
                        bands: ["B03", "B04", "B08", "SCL"],
                        units: "DN"
                    }],
                    output: {
                        bands: 4,
                        sampleType: "FLOAT32"
                    }
                };
            }

            function evaluatePixel(sample) {
                return [sample.B03, sample.B04, sample.B08, sample.SCL];
            }
        """,

        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(fecha, fecha),
            )
        ],

        responses=[
            SentinelHubRequest.output_response("default", MimeType.TIFF)
        ],

        bbox=bbox,
        size=bbox_to_dimensions(bbox, resolution=12),
        config=config
    )

    try:
        data = request.get_data()
        print(f"descargado: {lago_nombre} y {fecha}")
        return data[0]

    except Exception as e:
        print(f"error para: {lago_nombre} y {fecha}: {str(e)}")
        return None


print("descarga lista")

descarga lista
